In [2]:
import sys
sys.path.insert(0, "../../run")
from run_config import REPO_PATH, SEASONS, DATA_PROCESSOR_PATH
sys.path.insert(1, f"{REPO_PATH}")

from typing import List
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

import joblib
from collections import defaultdict

# Read Processed Data

In [3]:
processed_data_path='../../data/processed/premier_league/'

In [4]:
seasons=sorted(SEASONS)

In [5]:
data_dfs=[pd.read_csv(f"{processed_data_path}/{season}/all_data_df.csv") for season in seasons]

# A class for reproducability

In [10]:
class TeamRestDaysCalculator:
    def __init__(self, home_col='home', away_col='away', date_col='date'):
        self.home_col = home_col
        self.away_col = away_col
        self.date_col = date_col

    def transform(self, season_dfs):
        all_results = []
        last_match_dates = defaultdict(lambda: None)

        for df in season_dfs:
            df = df.copy()
            df.sort_values(by=self.date_col, inplace=True)

            home_rest_days = []
            away_rest_days = []

            for _, row in df.iterrows():
                date = pd.to_datetime(row[self.date_col])
                home_team = row[self.home_col]
                away_team = row[self.away_col]

                last_home_date = last_match_dates[home_team]
                last_away_date = last_match_dates[away_team]

                home_rest = (date - last_home_date).days if last_home_date is not None else None
                away_rest = (date - last_away_date).days if last_away_date is not None else None

                home_rest_days.append(home_rest)
                away_rest_days.append(away_rest)

                last_match_dates[home_team] = date
                last_match_dates[away_team] = date

            df['days_since_last_home'] = home_rest_days
            df['days_since_last_away'] = away_rest_days
            all_results.append(df[['days_since_last_home', 'days_since_last_away']])

        return pd.concat(all_results, ignore_index=True)


In [11]:
team_rest_days_calculator = TeamRestDaysCalculator()
team_rest_days_features = team_rest_days_calculator.transform(data_dfs)

In [12]:
team_rest_days_features

,days_since_last_home,days_since_last_away
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN
...,...,...
1885,6.0,7.0
1886,7.0,6.0
1887,8.0,8.0
1888,8.0,8.0
